In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers wandb datasets
!pip install -q openenv-core 2>/dev/null || true


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)


In [ ]:
# Sanity check — verify reward.py works before training
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from reward import compute_reward

good = 'If you dropped a feather and a hammer in a vacuum, what would you expect to happen?'
bad  = 'The answer is that heavier objects fall faster due to gravity.'

r_good = compute_reward(good, 'falling objects', turn=1, understanding_score=0.1, history=[])
r_bad  = compute_reward(bad,  'falling objects', turn=1, understanding_score=0.1, history=[])

print(f'Good question reward: {r_good.total:+.2f}  ({r_good.feedback})')
print(f'Bad  answer  reward:  {r_bad.total:+.2f}  ({r_bad.feedback})')
assert r_good.total > 0, 'Good question should get positive reward'
assert r_bad.total  < 0, 'Direct answer should get negative reward'
print('Reward function OK ✅')


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import wandb
from reward import compute_reward, RewardResult
from models import SocraticAction
from server.environment import SocraticEnvironment
from students.scenarios import TRAINING_SCENARIOS

wandb.init(project='socratic-rl-training', resume='allow')

def socratic_reward(prompts, completions, **kwargs):
    """
    GRPO reward function.
    prompts:     list of prompt strings (one per sample)
    completions: list of generated strings (one per sample)
    Returns:     list of float rewards, one per completion.
    """
    rewards = []
    successes = 0

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        # Extract topic from prompt
        topic = ''
        for line in prompt.split('\n'):
            if line.startswith('Topic:'):
                topic = line.replace('Topic:', '').strip()
                break

        env = SocraticEnvironment(seed=1000 + i)
        scenario = TRAINING_SCENARIOS[i % len(TRAINING_SCENARIOS)]
        env._scenario = scenario
        env._history = []
        env._understanding = 0.0
        env._turn = 0

        total_reward = 0.0
        history = []
        # Each completion is one question (single-turn GRPO)
        question = completion.strip().splitlines()[0].strip()
        if not question:
            rewards.append(-1.0)
            continue

        # Score with reward.py
        result = compute_reward(
            output=question,
            topic=topic or scenario.topic,
            turn=env._turn,
            understanding_score=env._understanding,
            history=history,
        )
        total_reward += result.total

        # Also step the environment to get understanding signal
        obs = env.step(SocraticAction(question=question))
        if obs.done and env._understanding >= 0.90:
            successes += 1
            total_reward += 0.5  # bonus for completing the episode

        rewards.append(total_reward)

    mean_reward = sum(rewards) / max(len(rewards), 1)
    success_rate = successes / max(len(rewards), 1)
    wandb.log({'mean_reward': mean_reward, 'success_rate': success_rate})
    print(f'  mean_reward={mean_reward:.3f}  success_rate={success_rate:.2%}')
    return rewards


In [ ]:
from datasets import Dataset
from students.scenarios import TRAINING_SCENARIOS

rows = []
for _ in range(50):
    for s in TRAINING_SCENARIOS:
        prompt = (
            'You are a Socratic tutor.\n'
            f'Topic: {s.topic}\n'
            f'Student misconception: {s.misconception}\n'
            'Rules:\n'
            '1) Ask questions only — never state the answer.\n'
            '2) Be specific to the topic.\n'
            '3) End your question with a ?.\n'
            'Your Socratic question:'
        )
        rows.append({'prompt': prompt})

dataset = Dataset.from_list(rows)
print(f'Dataset size: {len(dataset)} prompts')
print('Example prompt:\n', dataset[0]['prompt'])


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from dynamic_curriculum import CurriculumScheduler

# Curriculum scheduler — automatically promotes easy→medium→hard
# as the agent's success rate improves
curriculum = CurriculumScheduler(seed=42)

training_args = GRPOConfig(
    output_dir='./socratic-agent',
    num_generations=4,           # 4 completions per prompt, GRPO ranks them
    max_completion_length=128,   # one question, doesn't need to be long
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch = 8
    learning_rate=5e-6,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    report_to='wandb',
    fp16=True,                   # needed on T4
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[socratic_reward],
    args=training_args,
    train_dataset=dataset,
)

print('Starting GRPO training...')
trainer.train()
print('Training complete.')


In [ ]:
import os

# Save locally
model.save_pretrained('socratic-agent-final')
tokenizer.save_pretrained('socratic-agent-final')
print('Saved to ./socratic-agent-final')

# Push to HuggingFace Hub
# Replace with your HF username before running
HF_USERNAME = os.environ.get('HF_USERNAME', 'your-username')
repo_id = f'{HF_USERNAME}/socratic-rl-agent'
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print(f'Pushed to https://huggingface.co/{repo_id}')

# Run eval
print('\nRunning evaluation...')
!python eval.py --model_path ./socratic-agent-final --n_episodes 30
